<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Declare a folder's fields. Tag documents as you upload them.**

Tags carry what a file does not say about itself: which team owns it, what kind of document it is, which year it belongs to. This notebook demonstrates how to declare a folder's metadata fields on PageIndex Cloud and attach **your own metadata** to documents as you upload them.

We declare a folder's metadata fields, `type`, `team`, and `year`, upload four documents into it, each tagged with its values, and read the tags back. Once a folder's fields are declared, the chat agent can filter, sort, and count its documents by them; see [PageIndex Metadata Search](https://github.com/VectifyAI/PageIndex/blob/main/cookbook/pageindex-metadata-search.ipynb) for chat over a large tagged library.

1. **Declare the fields** — Give a folder its fields with `create_metadata_schema()`.
2. **Tag on upload** — Pass a `metadata` dict to `submit_document()`.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) ready. This demo creates a folder in your PageIndex Cloud library and uploads four small PDFs into it.


## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the demo documents.


In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your PageIndex API key. The client stores your documents and their metadata on PageIndex Cloud.


In [2]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"

client = PageIndexClient(index="cloud")

## 02 · Declare the fields

Create a top-level folder and declare its fields. Each field has a type, `string`, `number`, or `boolean`, and a description that tells the agent what it holds. Subfolders share the fields. Metadata needs declared fields: PageIndex Cloud rejects tags on a document whose folder has none. A folder's fields are set once; to change them, call `delete_metadata_schema()` and declare them again, which removes the tags the new fields don't accept. A folder left by an earlier run is reused, and `get_metadata_schema()` shows whether it already has fields.


In [3]:
from pageindex import PageIndexAPIError

FOLDER = "Metadata Demo"
FIELDS = {
    "type": {"type": "string", "description": "Document kind: earnings_release, regulation, or paper"},
    "team": {"type": "string", "description": "Team that owns the document"},
    "year": {"type": "number", "description": "Year the document was published"},
}

try:
    folder_id = client.get_folder_id(FOLDER)  # created on an earlier run
except PageIndexAPIError:
    folder_id = client.create_folder(FOLDER)["folder"]["id"]

if client.get_metadata_schema(folder_id)["schema"] is None:
    client.create_metadata_schema(folder_id, FIELDS)
print(f"{FOLDER}: {folder_id}")

Metadata Demo: cmvi17aah00040ap9cfez1efj


## 03 · Tag documents on upload

Download four PDFs from the PageIndex repository and submit each one into the folder with its tags. `team` is something no file states about itself; only the tag knows it. `wait=True` returns once the document is ready. A document already in the folder from an earlier run is reused rather than uploaded again.


In [4]:
import requests

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "q1-fy25-earnings.pdf": {"type": "earnings_release", "team": "finance", "year": 2025},
    "Regulation Best Interest_Interpretive release.pdf": {"type": "regulation", "team": "legal", "year": 2019},
    "earthmover.pdf": {"type": "paper", "team": "research", "year": 2013},
    "attention-residuals.pdf": {"type": "paper", "team": "research", "year": 2026},
}

existing = {doc["name"]: doc["id"] for doc in client.list_documents(folder_id=folder_id)["documents"]}

doc_ids = {}
for file_name, metadata in DOCS.items():
    if file_name in existing:  # uploaded by an earlier run
        doc_ids[file_name] = existing[file_name]
        print(f"{file_name}: {doc_ids[file_name]} (already uploaded)")
        continue

    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(
        file_name, wait=True, folder_id=folder_id, metadata=metadata,
    )["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

q1-fy25-earnings.pdf: pi-cmvi1etih00050bp9uh6viy63
Regulation Best Interest_Interpretive release.pdf: pi-cmvi1fysj00040apbh630pbwe
earthmover.pdf: pi-cmvi1grw300020dpbm3eknt7l
attention-residuals.pdf: pi-cmvi1hl1900080cpb8cgysmub


The tags come back under `metadata` from `get_document()`, and on every document from `list_documents()`.


In [5]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

q1-fy25-earnings.pdf: {'team': 'finance', 'type': 'earnings_release', 'year': 2025}
Regulation Best Interest_Interpretive release.pdf: {'team': 'legal', 'type': 'regulation', 'year': 2019}
earthmover.pdf: {'team': 'research', 'type': 'paper', 'year': 2013}
attention-residuals.pdf: {'team': 'research', 'type': 'paper', 'year': 2026}


Metadata is a flat dict of `str`, `int`, `float`, or `bool` values (numbers must be finite, so no `NaN` or `Infinity`), with keys that start with a letter and contain only letters, digits, and underscores. Its serialized size is at most 8 KB. A document may use only the fields declared on its folder, or on your account with `folder_id="root"` when the folder declares none, each with its declared type; with no fields declared, any metadata is rejected. PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See the [metadata documentation](https://docs.pageindex.ai/sdk/documents#metadata) for details.


---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
